# H-mode dynamics
This notebook demonstrates using the H-mode dynamics module.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import diffrax
import jax.numpy as jnp

from popsim.modules.hmode_dynamics import HmodeDynamics
from popsim.simulate import make_time_base, simulate
from popsim.visualize import visualize_time_series

state = HmodeDynamics.State(hmode=jnp.array(0.0))
lh_threshold_MW = 10.0
hl_threshold_MW = 7.0

#
# Four phases:
#   1) Conducted power well above LH threshold.
#   2) Conducted power above HL threshold but below LH threshold.
#   3) Conducted power well below HL threshold.
#   4) Conducted power above LH threshold, but input power below LH threshold.
#

times = jnp.array([0.0, 0.5, 0.6, 1.0, 1.01, 1.11, 1.3])
conducted_powers = jnp.array([15.0, 15.0, 9.0, 9.0, 7.0, 5.0, 15.0])
conducted_powers_traj = diffrax.LinearInterpolation(ts=times, ys=conducted_powers)
input_powers = jnp.array([15.0, 15.0, 15.0, 15.0, 15.0, 0.0, 0.0])
input_powers_traj = diffrax.LinearInterpolation(ts=times, ys=input_powers)

# Set params
params = HmodeDynamics.Params(
    transition_characteristic_time=0.1,
    P_tau_MW=conducted_powers_traj,
    P_input_MW=input_powers_traj,
    hl_threshold_MW=hl_threshold_MW,
    lh_threshold_MW=lh_threshold_MW,
)

In [ ]:
time_base = make_time_base(t0=0.0, t1=10.0, dt=0.01)
hmode_module = HmodeDynamics(config=HmodeDynamics.Config())
print(hasattr(state, "keys"))

sol_xarray = simulate(hmode_module, time_base, state, params)

print(sol_xarray)
visualize_vars = [k for k in sol_xarray.keys() if k.startswith("state.")]
print(visualize_vars)
visualize_time_series(sol_xarray[visualize_vars])